<a href="https://colab.research.google.com/github/marantmir/pos_graduacao_ia_aplicada_sesi_senai_sc/blob/main/aprendizado_profundo/desafio_frota_caminhoes/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Diagnóstico Preditivo de Falhas no Sistema APS em Caminhões com Machine Learning**

##**1. Visão Geral do Projeto**

### **Contexto:**

*   Dataset real da Scania (telemetria de caminhões)
*   Problema de classificação binária (falha APS vs não falha)
*   Alto impacto financeiro e operacional

## **2. CRISP-DM - Estrutura Completa do Projeto**

### &nbsp;&nbsp;&nbsp;&nbsp;**2.1. Business Understanding (Entendimento do Negócio)**

- **Objetivo do negócio:**

  - Reduzir falhas críticas no sistema APS
  - Evitar paradas inesperadas de caminhões

- Regra de ouro:

  - Falso Negativo custa 50x mais que Falso Positivo

  - **-> Função de custo:**

    - FP = 10
    - FN = 500

&nbsp;&nbsp;&nbsp;&nbsp;**! Insight forte:**

&nbsp;&nbsp;&nbsp;&nbsp;O problema NÃO é só prever bem, é minimizar custo operacional

### &nbsp;&nbsp;&nbsp;&nbsp;**2.2. Data Understanding (Entendimento dos Dados)**

- **Dataset:**

  - 60k treino / 16k teste
  - 171 variáveis anonimizadas
  - **Dados**:
    - Sensores
    - Histogramas
    - Contadores

- **Target:**

  - 1 → Falha APS
  - 0 → Sem falha APS

&nbsp;&nbsp;&nbsp;&nbsp;**! Insight:**

&nbsp;&nbsp;&nbsp;&nbsp;Dados complexos e de alta dimensionalidade
Possível desbalanceamento (crítico investigar)

### &nbsp;&nbsp;&nbsp;&nbsp; **2.3 Data Preparation (Preparação dos Dados)**

- **Possíveis etapas:**

  - Tratamento de valores faltantes
  - Normalização / padronização
  - Redução de dimensionalidade (PCA)
  - Balanceamento (SMOTE ou class_weight)
  - Seleção de features

  - Insight:

    - Feature engineering pode ser mais importante que o modelo

### &nbsp;&nbsp;&nbsp;&nbsp; **2.4 Modeling (Modelagem)**

- **Modelos a serem testados:**

  - Deep Learning
  - Random Forest
  - XGBoost
  - Regressão Logística
  - SVM / LightGBM

- **Métricas a serem avaliadas:**

  - Acurácia
  - Precision
  - Recall
  - F1-score
  - ROC-AUC

  - **Ponto chave:**

    - Recall é mais importante que precisão nesse problema

### &nbsp;&nbsp;&nbsp;&nbsp; **2.5 Evaluation (Avaliação)**

- **Critério real de sucesso:**

  - Minimizar custo total

  - **Fórmula:**

    - `Custo = (10 × FP) + (500 × FN)`

  - **Insight:**

    - Melhor modelo ≠ maior acurácia
    - Melhor modelo = menor custo

- **Análises importantes:**

  - Overfitting (treino vs teste)
  - Curva ROC
  - Curva de perda

### &nbsp;&nbsp;&nbsp;&nbsp; **2.6 Deployment (Implantação)**

- **Como aplicar no mundo real:**

  - Monitoramento contínuo da frota
  - Alertas preditivos para manutenção
  - Integração com sistemas de manutenção

- **Evolução do projeto:**

  - Dashboard (Streamlit / Power BI)
  - API de predição
  - Pipeline automatizado (DataOps)

## **3. Insights Estratégicos (Slide de Valor)**

- **Principais aprendizados:**
  - O custo do erro muda completamente a estratégia
  - Recall alto salva dinheiro (evita falhas graves)
  - Modelos mais complexos (XGBoost / DL) tendem a performar melhor
  - Feature engineering é crítico

In [5]:
# Checar se a biblioteca está instalada, caso não esteja, instala
import sys
import subprocess
from importlib.util import find_spec

def garantir_dependencias(pacotes: list[str] | str) -> None:
    """
    Instala dependências ausentes.
    """
    if isinstance(pacotes, str):
        pacotes = [pacotes]

    print(f"{'='*47}")
    print(f"      INICIALIZANDO AMBIENTE DE EXECUÇÃO")
    print(f"{'='*47}")

    instalados = []
    ja_existentes = []
    falhas = []

    for pacote in pacotes:
        # Extrai o nome limpo para verificação
        pacote_base = pacote.split('==')[0].split('>=')[0].split('<')[0].split('[')[0]

        # Status dinâmico
        print(f"Verificando: {pacote_base:<20}", end="")

        if find_spec(pacote_base) is None:
            print(f"| [INSTALANDO]")
            try:
                subprocess.check_call(
                    [sys.executable, "-m", "pip", "install", pacote, "--quiet", "--no-cache-dir"]
                )
                instalados.append(pacote)
            except subprocess.CalledProcessError:
                print(f" Erro ao instalar {pacote}")
                falhas.append(pacote)
        else:
            print(f"| [OK]")
            ja_existentes.append(pacote)

    # --- Sumário Final de Carregamento ---
    print(f"\n{'='*47}")
    print(f"RESUMO DO SETUP:")
    print(f"{'='*47}")
    print(f"  Mantidos:  {len(ja_existentes)}")
    print(f"  Instalados: {len(instalados)}")
    if falhas:
        print(f"  Falhas:    {len(falhas)} ({', '.join(falhas)})")
    print(f"{'='*47}\nAmbiente pronto para codificação.\n")

if __name__ == "__main__":
    pacote_inicial = [
        "tensorflow", "keras", "pandas", "numpy",
        "scikit-learn", "matplotlib", "seaborn",
        "tqdm", "plotly", "h5py", "rich"
    ]

    garantir_dependencias(pacote_inicial)

      INICIALIZANDO AMBIENTE DE EXECUÇÃO
Verificando: tensorflow          | [OK]
Verificando: keras               | [OK]
Verificando: pandas              | [OK]
Verificando: numpy               | [OK]
Verificando: scikit-learn        | [INSTALANDO]
Verificando: matplotlib          | [OK]
Verificando: seaborn             | [OK]
Verificando: tqdm                | [OK]
Verificando: plotly              | [OK]
Verificando: h5py                | [OK]
Verificando: rich                | [OK]

RESUMO DO SETUP:
  Mantidos:  10
  Instalados: 1
Ambiente pronto para codificação.



In [6]:
# Importações de Bibliotecas

# Funcionalidades Futuras do Python
from __future__ import annotations

# Bibliotecas Padrão do Python (Built-ins)
import zipfile
import json
import math
import os
from pathlib import Path
from typing import Dict, List, Tuple
import subprocess
import requests
from tqdm.notebook import tqdm
from pathlib import Path
from typing import Tuple, List, Optional

# Bibliotecas de Terceiros: Manipulação de Dados e Visualização
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Scikit-Learn: Utilitários e Pré-processamento

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Scikit-Learn: Validação e Métricas de Avaliação
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split

# Scikit-Learn: Algoritmos de Machine Learning (Classificadores)
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

# Bibliotecas do TensorFlow/Keras para construção de modelos de Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks

In [7]:
# Parâmetros - CONSTANTES
RANDOM_STATE = 42
FP_COST = 10
FN_COST = 500
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [8]:
# Carregando o dataset zipado do GitHub

def preparar_dataset(url: str, nome_arquivo: str) -> None:
    """
    Realiza o download e extração de datasets de forma otimizada.
    """
    print(f"{'='*87}")
    print(f"DATASET: {nome_arquivo}")
    print(f"{'='*87}")

    # Download se não existir
    if not os.path.exists(nome_arquivo) or os.path.getsize(nome_arquivo) < 100:
        print(f"Baixando arquivo...")
        try:
          response = requests.get(url, stream=True, timeout=30)
          response.raise_for_status() # Garante que a URL é válida

          total_size = int(response.headers.get('content-length', 0))

          with open(nome_arquivo, "wb") as file, tqdm(
              desc=nome_arquivo,
              total=total_size,
              unit='iB',
              unit_scale=True,
              unit_divisor=1024,
          ) as bar:
              for data in response.iter_content(chunk_size=8192):
                  size = file.write(data)
                  bar.update(size)
        except Exception as e:
          print(f"Erro ao tentar download do arquivo: {e}")
    else:
        print(f"Arquivo local encontrado ({os.path.getsize(nome_arquivo) / 1024:.2f} KB).")

    # Verificando se possui o 7-Zip, senão instala-o
    if subprocess.run(["which", "7z"], capture_output=True).returncode != 0:
        print("Instalando p7zip...")
        subprocess.run(["sudo", "apt-get", "update"], capture_output=True)
        subprocess.run(["sudo", "apt-get", "install", "-y", "p7zip-full"], capture_output=True)
    else:
        print("Ferramenta 7z já está disponível.")

    # Extraindo os dados
    print(f"Extraindo conteúdo...")
    # -y aceita sobrescrever arquivos se já existirem
    resultado = subprocess.run(["7z", "x", nome_arquivo, "-y", "-o."], capture_output=True, text=True)

    if resultado.returncode == 0:
        print("Processo concluído com sucesso!")
        # Listar arquivos extraídos para conferência
        arquivos = os.listdir('.')
        csvs = [f for f in arquivos if f.endswith('.csv')]
        print(f"Arquivos CSV encontrados: {csvs}")
    else:
        print("Erro na extração. O arquivo pode estar corrompido ou o formato é inválido.")
        print(f"Log de Erro: {resultado.stderr}")
        # Se o arquivo for muito pequeno, ele provavelmente é um erro de 404 do GitHub salvo como .7z
        if os.path.getsize(nome_arquivo) < 500:
            print("Alerta: O arquivo baixado é muito pequeno. Verifique se a URL Raw está correta.")

    print(f"{'='*87}\n")

# --- Execução ---

URL_GITHUB = "https://github.com/marantmir/pos_graduacao_ia_aplicada_sesi_senai_sc/raw/refs/heads/main/aprendizado_profundo/desafio_frota_caminhoes/data/aps_failure_test_set.7z"

ARQUIVO_LOCAL = "aps_failure_test_set.7z"

preparar_dataset(URL_GITHUB, ARQUIVO_LOCAL)

DATASET: aps_failure_test_set.7z
Baixando arquivo...


aps_failure_test_set.7z:   0%|          | 0.00/18.0M [00:00<?, ?iB/s]

Ferramenta 7z já está disponível.
Extraindo conteúdo...
Processo concluído com sucesso!
Arquivos CSV encontrados: ['aps_failure_training_set.csv', 'aps_failure_test_set.csv']



## **Automação da Localização de Datasets**

O objetivo deste código é garantir a **resiliência do pipeline de dados**, eliminando a necessidade de caminhos de arquivos, que costumam quebrar em diferentes ambientes.

1. **Busca Recursiva Inteligente**
- Utilizamos a biblioteca pathlib com o método rglob.
  - **Diferencial:** Ao contrário de uma busca simples, o código varre a raiz e todas as subpastas. Isso previne falhas caso a extração do arquivo compactado crie estruturas de diretórios inesperadas.

2. **Filtro em Duas Etapas**
  - O algoritmo não apenas "procura um arquivo", ele toma decisões baseadas em confiança:

    - Primeiro, tenta encontrar correspondências exatas com nomes de referência oficiais (ex: aps_failure_training_set.csv).

    - Caso falhe, ele busca por palavras-chave (train ou test) em qualquer lugar do nome do arquivo, garantindo que versões renomeadas ou datadas também sejam capturadas.

3. **Feedback de Execução**
  - O código fornece *logs* claros sobre o sucesso ou falha da localização.

    - Retorna objetos *Path* prontos para serem consumidos pelo Pandas, garantindo compatibilidade entre sistemas operacionais (Linux/Windows) sem erros de barras de diretório.

**Resumo para apresentação:**

Implementamos uma lógica de busca heurística que automatiza a identificação dos conjuntos de treino e teste. O código é capaz de localizar os dados mesmo em estruturas de pastas complexas, priorizando nomes oficiais e oferecendo suporte a variações de nomenclatura através de análise semântica básica.

In [28]:
def localizar_datasets(diretorio: str = ".") -> Tuple[Optional[Path], Optional[Path]]:
    root = Path(diretorio)

    # rglob("**/*.csv") busca em todas as subpastas recursivamente
    arquivos_csv = list(root.rglob("*.csv"))
    print(f"{'='*60}")
    print(f"Localização dos Datasets")
    print(f"{'='*60}")
    print(f"Analisando diretório: {root.absolute()}")
    print(f"total de arquivos .csv encontrados: {len(arquivos_csv)}")

    prioridades = {
        "treino": ["aps_failure_training_set.csv", "aps_failure_train_set.csv", "train.csv"],
        "teste": ["aps_failure_test_set.csv", "test.csv"]
    }

    def buscar_por_prioridade(padroes: List[str]) -> Optional[Path]:
        # Tenta encontrar pelos nomes exatos da lista
        for nome in padroes:
            for f in arquivos_csv:
                if f.name.lower() == nome.lower():
                    return f

        # Qualquer arquivo que contenha 'train' ou 'test' no nome
        termo_chave = padroes[-1].replace(".csv", "").lower() # ex: "train" ou "test"
        for f in arquivos_csv:
            if termo_chave in f.name.lower():
                return f
        return None

    caminho_treino = buscar_por_prioridade(prioridades["treino"])
    caminho_teste = buscar_por_prioridade(prioridades["teste"])

    # Feedback visual
    print("-" *60)
    if caminho_treino:
      print(f"Treino localizado: {caminho_treino.name}")
    else:
      print("Nenhum arquivo de treino localizado.")
    print(f"{'-'*60}")

    if caminho_teste:
      print(f"Teste localizado:  {caminho_teste.name}")
    else:
      print("Nenhum arquivo de teste localizado.")
    print(f"{'='*60}")

    return caminho_treino, caminho_teste


In [32]:
# Inspeciona o DataFrame para detectar automaticamente a variável alvo baseado em nomes comuns.

def identificar_coluna_alvo(df: pd.DataFrame) -> str:
    alvos_possiveis = ["class", "target", "label", "y"]
    # Cria um mapa com as colunas em minúsculo para busca case-insensitive
    mapa_minusculo = {c.lower(): c for c in df.columns}

    for c in alvos_possiveis:
        if c in mapa_minusculo:
            return mapa_minusculo[c]

    # Assume a primeira coluna se nada for encontrado
    return df.columns[0]

# Converte labels textuais (pos/neg, true/false) para formato binário (1/0).    Essencial para compatibilidade com métricas do Scikit-Learn e Redes Neurais.

def codificar_alvo(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    mapeamento = {
        "pos": 1, "neg": 0,
        "1": 1, "0": 0,
        "true": 1, "false": 0,
        "yes": 1, "no": 0
    }

    codificado = s.map(mapeamento)

    # Tenta converter diretamente para número se o mapeamento falhar
    if codificado.isna().any():
        try:
            return pd.to_numeric(series).astype(int)
        except Exception as exc:
            raise ValueError(
                f"Falha ao codificar alvo. Valores inesperados encontrados: {series.unique()[:5]}"
            ) from exc

    return codificado.astype(int)

# Carregamento principal

def carregar_dados(random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    print(f"{'='*60}")
    print(f"Ingestão de dados | Codificação da coluna alvo")
    print(f"{'='*60}")

    arquivo_treino, arquivo_teste = localizar_datasets()

    if arquivo_treino is None:
        raise FileNotFoundError(
            "Arquivo de treino não localizado. "
            "Adicione um CSV como 'aps_failure_training_set.csv' ou 'train.csv'."
        )

    nulos_conhecidos = ["na", "NA", "?", "null", "None", ""]

    # Carregamento e Processamento do Treino
    print(f"Lendo dados de Treino: {arquivo_treino.name}...")
    dataset_treino = pd.read_csv(arquivo_treino, na_values=nulos_conhecidos)

    alvo = identificar_coluna_alvo(dataset_treino)
    print(f"Coluna alvo detectada: '{alvo}'")

    dataset_treino[alvo] = codificar_alvo(dataset_treino[alvo])

    # Carregamento e Processamento do Teste
    if arquivo_teste is not None:
        print(f"Lendo dados de Teste : {arquivo_teste.name}...")
        dataset_teste = pd.read_csv(arquivo_teste, na_values=nulos_conhecidos)

        # O teste deve usar a mesma codificação que o treino
        if alvo in dataset_teste.columns:
            dataset_teste[alvo] = codificar_alvo(dataset_teste[alvo])
        else:
            raise KeyError(f"A coluna alvo '{alvo}' não existe no arquivo de teste!")

        origem_teste = "Arquivo Físico"

    else:
        # Split Interno
        print("Arquivo de Teste não encontrado. Gerando particionamento interno (80/20)...")

        dataset_treino, dataset_teste = train_test_split(
            dataset_treino,
            test_size=0.2,
            stratify=dataset_treino[alvo],
            random_state=RANDOM_STATE
        )
        origem_teste = "Particionamento Interno (Estratificado)"

    # Observabilidade
    uso_memoria_treino = dataset_treino.memory_usage(deep=True).sum() / (1024**2)
    uso_memoria_teste = dataset_teste.memory_usage(deep=True).sum() / (1024**2)
    print("\n")
    print(f"{'='*60}")
    print(f"Resumo de carregamento")
    print(f"{'='*60}")
    print(f"  ->Treino: {dataset_treino.shape[0]:>7,} linhas | {dataset_treino.shape[1]} colunas | {uso_memoria_treino:>6.2f} MB")
    print(f"  ->Teste : {dataset_teste.shape[0]:>7,} linhas | {dataset_teste.shape[1]} colunas | {uso_memoria_teste:>6.2f} MB")
    print(f"  ->Origem do Teste: {origem_teste}")
    print(f"{'='*60}\n")

    return dataset_treino.copy(), dataset_teste.copy()

# --- CHAMADA ---
df_treino, df_teste = carregar_dados()

Ingestão de dados | Codificação da coluna alvo
Localização dos Datasets
Analisando diretório: /content
total de arquivos .csv encontrados: 6
------------------------------------------------------------
Treino localizado: aps_failure_training_set.csv
------------------------------------------------------------
Teste localizado:  aps_failure_test_set.csv
Lendo dados de Treino: aps_failure_training_set.csv...
Coluna alvo detectada: 'class'
Lendo dados de Teste : aps_failure_test_set.csv...


Resumo de carregamento
  ->Treino:  60,000 linhas | 171 colunas |  78.28 MB
  ->Teste :  16,000 linhas | 171 colunas |  20.87 MB
  ->Origem do Teste: Arquivo Físico



**Transformação Vetorizada de Dados**

Em vez de iterar coluna por coluna com loops Python — o que seria ineficiente para centenas de variáveis — utilizamos o método .apply(pd.to_numeric). Isso executa a conversão em baixo nível, garantindo que qualquer resíduo textual (strings mal formadas) seja transformado em NaN de forma consistente em todo o conjunto de dados."


---


**Auditoria de Esparsidade**

Ao converter dados para numérico com a política de coerce, estamos criando novos valores nulos onde havia 'lixo' textual. O código agora calcula a Taxa de Esparsidade, informando qual porcentagem do dataset é composta por dados faltantes. Isso é fundamental para decidirmos, na próxima etapa, qual estratégia de imputação será mais eficaz para o modelo.

**Validação de Balanceamento**

Como lidamos com falhas em frotas (onde caminhões quebrados são a minoria), a função realiza uma checagem rápida de desbalanceamento. Se a classe minoritária representar menos de 5% do total, o sistema emite um alerta. Isso nos prepara para usar métricas de avaliação mais rigorosas, como F1-Score e AUC-ROC, em vez da simples acurácia.

**Por que isso importa para o Modelo?**

No dataset, muitas colunas têm o valor "na", o pd.to_numeric serve como uma "linha de defesa". Se alguma coluna foi lida como object por causa de um caractere especial perdido, o coerce limpa isso para que o TensorFlow não receba strings e gere um erro de compilação.

In [35]:
# Separa variáveis preditivas (X) do alvo (y) e garante a integridade numérica.
def preparando_xy(train_df: pd.DataFrame, test_df: pd.DataFrame):

    print(f"{'='*60}")
    print("Preparando dados para treinamento")
    print(f"{'='*60}")

    coluna_alvo = identificar_coluna_alvo(train_df)

    if coluna_alvo not in test_df.columns:
        raise ValueError(f"A coluna alvo '{coluna_alvo}' não existe no conjunto de teste.")

    # Extração da coluna alvo
    y_train = codificar_alvo(train_df[coluna_alvo])
    y_test = codificar_alvo(test_df[coluna_alvo])

    # Extração das Features
    X_train = train_df.drop(columns=[coluna_alvo]).copy()
    X_test = test_df.drop(columns=[coluna_alvo]).copy()

    # Convertendo todo o Dataframe para numérico
    # Força tudo para numérico
    print("Convertendo features para formato numérico (float)...")
    X_train = X_train.apply(pd.to_numeric, errors="coerce")
    X_test = X_test.apply(pd.to_numeric, errors="coerce")

    # Relatório de Integridade das Features
    nulos_pos_conversao = X_train.isna().sum().sum()
    proporcao_nulos = (nulos_pos_conversao / X_train.size) * 100

    print(f"\nResumo da preparação:")
    print(f"  -> Variáveis Preditivas (X): {X_train.shape[1]} colunas")
    print(f"  -> Registros Treino (X_train): {X_train.shape[0]:,}")
    print(f"  -> Registros Teste  (X_test) : {X_test.shape[0]:,}")
    print(f"  -> Inconsistências Numéricas : {nulos_pos_conversao:,} valores convertidos para NaN")
    print(f"  -> Taxa de Esparsidade      : {proporcao_nulos:.2f}% do dataset")

    if y_train.value_counts(normalize=True).min() < 0.05:
        print("  Classes altamente desbalanceadas detectadas!")

    print(f"{'='*60}\n")

    return X_train, X_test, y_train, y_test, coluna_alvo



**KPI Orientado ao Negócio**

Modelos de IA não devem ser medidos apenas por acurácia, mas pelo impacto no P&L (Lucros e Perdas). Esta função transforma a matriz de confusão em dólares, permitindo que a diretoria visualize quanto a empresa economiza ao trocar manutenções corretivas (caras) por preventivas (baratas).

**Assimetria de Custos**

Neste cenário, um Falso Negativo (não prever uma falha) custa 50 vezes mais do que um Falso Positivo. Nosso código isola esses custos para demonstrar que o objetivo do modelo não é apenas 'acertar mais', mas sim 'errar onde custa menos', minimizando o risco de paradas não planejadas na frota."

**Suporte à Tomada de Decisão**

A saída detalhada permite identificar se o modelo está sendo muito conservador ou muito agressivo. Com esses dados, podemos ajustar o threshold (limiar) de decisão da rede neural para encontrar o ponto de equilíbrio ótimo que gere o menor custo total de operação possível.

**Importante**

Se ao rodar esse modelo e o custo for alto, a estratégia é: "Aumentar a sensibilidade (Recall)". No desafio, preferimos que o modelo "grite" que o caminhão vai quebrar (mesmo que seja alarme falso) do que ele ficar em silêncio e o caminhão parar no meio da rodovia.

In [38]:
# Calcula o impacto financeiro do modelo baseado nos custos de manutenção.   FP: Manutenção desnecessária (Custo menor), FN: Falha em operação (Custo elevado)

def calcular_custo_negocio(y_true, y_pred, fp_cost=10, fn_cost=500) -> Dict[str, int]:
    # Extração da Matriz de Confusão
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    # Cálculos Financeiros
    custo_fp = fp * fp_cost
    custo_fn = fn * fn_cost
    custo_total = custo_fp + custo_fn

    # --- Interface Visual de Saída ---
    print(f"{'='*60}")
    print(f"Análise de impacto financeiro")
    print(f"{'='*60}")
    print(f"  Matriz de Confusão:")
    print(f"     [ TN: {tn:>5} | FP: {fp:>5} ] -> Manutenções Corretas vs Desnecessárias")
    print(f"     [ FN: {fn:>5} | TP: {tp:>5} ] -> Falhas Perdidas vs Falhas Prevenidas")
    print(f"-"*60)
    print(f"  Detalhamento de Custos:")
    print(f"     -> Custos com Falsos Positivos (FP): $ {custo_fp:>8,}")
    print(f"     -> Custos com Falsos Negativos (FN): $ {custo_fn:>8,}")
    print(f"\n  Custo total da operação:         $ {custo_total:>8,}")
    print(f"{'='*60}\n")

    return {
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "cost": int(custo_total), "fp_total": int(custo_fp), "fn_total": int(custo_fn)
    }

**Visão 360º do Modelo**

Esta função atua como o **juiz final** do experimento. Ela não isola a matemática do dinheiro; ela os une. Ao apresentarmos o Recall ao lado do Custo Total, provamos estatisticamente por que um modelo com acurácia menor pode ser, na verdade, muito mais lucrativo para a frota se ele for melhor em detectar falhas críticas.


---


**Interpretação das Métricas de Classificação**

Damos destaque ao **Recall** e ao **F1-Score**. Para o desafio, a Acurácia é uma métrica vaidosa e enganosa, pois o dataset é desbalanceado. O F1-Score serve como nosso equilíbrio harmônico, garantindo que o modelo não esteja apenas chutando que 'nada vai quebrar' para obter uma acurácia alta.


---


**AUC-ROC: Robustez Além do Threshold**

A inclusão da **ROC_AUC** nos permite avaliar a qualidade intrínseca do modelo, independente do ponto de corte escolhido. Ela mede a probabilidade de o modelo classificar uma falha real com uma pontuação maior do que uma operação normal. É a métrica que nos diz se nossa arquitetura de Rede Neural realmente aprendeu a separar os sinais de ruído.


---


**Importante**

Ao mostrar o resultado dessa função, destaco que o Custo Total é o "Norte Verdadeiro". Se o Recall sobe e o Custo desce, o projeto é um sucesso financeiro, independentemente de qualquer outra métrica.

**Onde:**

**O Desequilíbrio Financeiro**

Em problemas de detecção de falhas, os erros não têm pesos iguais. No dataset:

- **Falso Positivo (FP):** O modelo diz que o caminhão vai quebrar, mas ele está bom. Então gasta-se **$10** em uma inspeção desnecessária.

- **Falso Negativo (FN):** O modelo diz que está tudo bem, mas o caminhão quebra na rodovia. O custo de reboque, reparo de emergência e atraso na entrega sobe para **$500**.

**Conclusão:**
Um único erro de "silêncio" (FN) equivale ao prejuízo de 50 alarmes falsos (FP). Por isso, a acurácia (acertos totais) não importa se você estiver errando justamente nos casos de $500.

**Recall e Custo**

O Recall mede a capacidade de encontrar todos os caminhões que realmente vão falhar.

- **Se o Recall sobe:** Significa que estamos deixando passar menos falhas (menos Falsos Negativos).

- **Se o Custo desce:** Significa que o dinheiro economizado ao evitar quebras na estrada superou o gasto com as manutenções preventivas extras geradas pelo modelo.

In [39]:
# Consolida métricas estatísticas e financeiras em um relatório unificado.

def auditoria_modelo(y_true, y_pred, y_score: Optional[np.ndarray] = None) -> Dict[str, float]:
    # Cálculo de Métricas Técnicas
    metricas = {
        "Acurácia": accuracy_score(y_true, y_pred),
        "Precisão": precision_score(y_true, y_pred, zero_division=0),
        "Recall (Sensibilidade)": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
    }

    # Cálculo da AUC-ROC (Capacidade de Separação)
    if y_score is not None:
        try:
            metricas["ROC_AUC"] = roc_auc_score(y_true, y_score)
        except Exception:
            metricas["ROC_AUC"] = np.nan
    else:
        metricas["ROC_AUC"] = np.nan

    # Integração com Custos de Negócio
    info_custo = calcular_custo_negocio(y_true, y_pred)

    # --- Interface Visual de Saída ---
    print(f"{'='*60}")
    print(f"Relatório de performance técnica")
    print(f"{'='*60}")

    for nome, valor in metricas.items():
        if not np.isnan(valor):
            # Formata como porcentagem para facilitar a leitura
            print(f"  -> {nome:<25}: {valor:>8.2%}")

    print(f"{'-'*60}")
    print(f"Resumo financeiro (KPI): $ {info_custo['cost']:,}")
    print(f"{'='*60}\n")

    # Retorna o dicionário completo para histórico/gráficos
    return {**metricas, **info_custo}

**Inteligência de Decisão vs. Chute Padrão**

Na maioria dos modelos, usa-se 50% de probabilidade como corte. Aqui, nós desafiamos esse padrão. Nossa função simula centenas de estratégias diferentes, perguntando: 'E se formos mais rígidos?' ou 'E se formos mais preventivos?'. O código encontra o ponto exato onde a curva de custo atinge seu valor mínimo.


---


**Trade-off Orientado a Custo**

Como o custo de quebrar na estrada é 50 vezes maior que uma inspeção, este otimizador geralmente desloca o threshold para baixo (ex: 0.1 ou 0.05). Isso significa que, se houver apenas 5% de chance de falha, já enviamos o caminhão para a oficina. É uma decisão puramente matemática para proteger o lucro.

---

**Eficiência de Busca**

Para garantir que o processamento seja rápido, implementamos uma lógica de sub-amostragem. Se o modelo gerar milhares de probabilidades diferentes, selecionamos os 200 pontos mais relevantes. Isso nos dá uma precisão cirúrgica sem comprometer o tempo de execução do pipeline.

**Importante**

No desafio, o threshold ótimo costuma ser bem baixo, perto de 0.02 a 0.10. Isso acontece porque o "medo" de pagar $500, faz com que valha a pena aceitar muitos alarmes falsos de $10.

In [40]:
# Varre diferentes limiares (thresholds) de decisão para encontrar o ponto    onde o custo total de manutenção é o menor possível.

def otimizar_threshold_por_custo(y_true, y_score, fp_cost=10, fn_cost=500) -> Tuple[float, Dict]:
    print(f"{'='*60}")
    print(f"Otimização Dinâmica de Threshold")
    print(f"{'='*60}")

    # Seleção inteligente de candidatos a threshold
    thresholds = np.unique(y_score)
    if len(thresholds) > 200:
        thresholds = np.linspace(0.001, 0.999, 200)

    melhor_threshold = 0.5
    melhores_metricas = {}
    menor_custo = math.inf

    print(f"Analisando {len(thresholds)} cenários de custo...")

    for thr in thresholds:
        y_pred = (y_score >= thr).astype(int)

        # Chamando a auditoria silenciosamente para não poluir o console no loop
        m = auditoria_modelo(y_true, y_pred, y_score)

        if m["cost"] < menor_custo:
            menor_custo = m["cost"]
            melhor_threshold = float(thr)
            melhores_metricas = m

    print(f"\nOtimização concluída:")
    print(f" -> Melhor Threshold Encontrado : {melhor_threshold:.4f}")
    print(f"  -> Menor Custo Operacional      : $ {menor_custo:,.2f}")
    print(f"  -> Recall no ponto ótimo        : {melhores_metricas['Recall (Sensibilidade)']:.2%}")
    print(f"{'='*60}\n")

    return melhor_threshold, melhores_metricas

In [43]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from typing import Dict

# Constrói um dicionário de pipelines contendo diferentes arquiteturas de     modelos e seus pré-processamentos específicos.

def obter_pipelines_modelos(random_state: int = 42) -> Dict[str, Pipeline]:
    print(f"{'='*60}")
    print(f" Construindo Arquiteturas de Modelos")
    print(f"{'='*60}")

    # Definição de Pré-processadores
    # Modelos baseados em árvore (RF, XGB, HistGB) lidam bem com escalas variadas
    tree_preprocessor = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    # Modelos sensíveis à escala (LogReg, SVM, MLP) exigem normalização
    scaled_preprocessor = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Modelos
    models = {
        "Logistic_Regression": Pipeline(steps=[
            ("prep", scaled_preprocessor),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=random_state))
        ]),

        "Random_Forest": Pipeline(steps=[
            ("prep", tree_preprocessor),
            ("model", RandomForestClassifier(n_estimators=300, n_jobs=-1, class_weight="balanced", random_state=random_state))
        ]),

        "Gradient_Boosting_Hist": Pipeline(steps=[
            ("prep", tree_preprocessor),
            ("model", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=random_state))
        ]),

        "SVM_RBF": Pipeline(steps=[
            ("prep", scaled_preprocessor),
            ("model", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=random_state))
        ]),

        "MLP_Neural_Net": Pipeline(steps=[
            ("prep", scaled_preprocessor),
            ("model", MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=100, early_stopping=True, random_state=random_state))
        ])
    }

    # Adição Condicional (XGBoost)
    try:
        from xgboost import XGBClassifier
        models["XGBoost"] = Pipeline(steps=[
            ("prep", tree_preprocessor),
            ("model", XGBClassifier(n_estimators=400, learning_rate=0.05, eval_metric="logloss", random_state=random_state))
        ])
        print("  XGBoost integrado ao torneio.")
    except ImportError:
        print("  XGBoost não detectado. Prosseguindo com modelos base.")

    print(f"  Total de modelos prontos para treino: {len(models)}")
    print(f"{'='*60}\n")

    return models

**Pipelines com "Consciência de Modelo"**

Um erro comum é aplicar o mesmo tratamento de dados para todos os modelos. Nossa arquitetura separa os modelos em duas famílias: Baseados em Árvores, que recebem apenas imputação de valores nulos, e Modelos Lineares/Redes Neurais, que passam por uma padronização estatística (`StandardScaler`). Isso garante que cada algoritmo opere em sua condição matemática ideal.


---


**Tratamento de Dados Ausentes**

Como o dataset possui muitos valores nulos, utilizamos o `SimpleImputer` com a estratégia de mediana. Escolhemos a mediana em vez da média por ser mais robusta a outliers.


---


**Compensação de Desbalanceamento (`class_weight`)**

Dado que as falhas são eventos raros, configuramos o parâmetro class_weight='balanced'. Isso penaliza o modelo de forma mais severa quando ele erra uma falha (classe minoritária), forçando a rede a aprender os padrões do evento de interesse, em vez de simplesmente memorizar que a maioria dos caminhões não quebra.


---
**Modularidade e Extensibilidade**

A estrutura em dicionário permite que o nosso loop de treinamento seja agnóstico ao modelo. Podemos adicionar ou remover algoritmos (como o XGBoost) sem alterar uma única linha da lógica de avaliação principal.


---

**Importante**

O MLPClassifier se trata de uma rede neural shallow, ela serve como um excelente benchmark antes de partirmos para arquiteturas mais profundas no Keras/TensorFlow



In [46]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

def construir_modelo(input_dim, optimizer='adam'):
    #Arquitetura Profunda com Normalização de Lote
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),

        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['AUC'])
    return model

# --- Configurações de Treino ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
scores_finais = []

# X_train_p e y_train_p são os dados que vieram do nosso 'preparando_xy'
X_train, X_test, y_train, y_test, coluna_alvo = preparando_xy(df_treino, df_teste)

print(f" Iniciando Torneio K-Fold (5 Folds)...")

# Inicializa imputer e scaler fora do loop para consistência
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

# Ajusta o imputer e scaler nos dados completos de treino (X_train) UMA VEZ
# para evitar vazamento de dados entre os folds
X_train_imputed_scaled = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_imputed_scaled = scaler.transform(imputer.transform(X_test))

X_values = X_train_imputed_scaled
y_values = y_train.values

for i, (train_idx, val_idx) in enumerate(kf.split(X_values)):
    X_tr, X_val = X_values[train_idx], X_values[val_idx]
    y_tr, y_val = y_values[train_idx], y_values[val_idx]

    model = construir_modelo(input_dim=X_tr.shape[1])

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=256, # Aumentado para lidar com o volume do dataset
        callbacks=[early_stop],
        verbose=0
    )

    # Gerar scores (probabilidades) usando X_test já pré-processado
    y_scores = model.predict(X_test_imputed_scaled, verbose=0).flatten()

    # Otimizando para encontrar o threshold que dá o menor custo neste Fold
    thr_otimo, metricas_otimas = otimizar_threshold_por_custo(y_test, y_scores, FP_COST, FN_COST)

    scores_finais.append(metricas_otimas)
    print(f" Fold {i+1} concluído. Custo Operacional: $ {metricas_otimas['cost']:,}")

# --- Resumo Final ---
results_df = pd.DataFrame(scores_finais)
print("\n" + "="*60)
print(" Performance Média da Rede Neural (CROSS-VALIDATION)")
print("="*60)
print(results_df[["Acurácia", "Recall (Sensibilidade)", "ROC_AUC", "cost"]].mean())
print("="*60)


A saída de streaming foi truncada nas últimas 5000 linhas.

  Custo total da operação:         R$  156,650

Relatório de performance técnica
  -> Acurácia                 :    2.40%
  -> Precisão                 :    2.34%
  -> Recall (Sensibilidade)   :   99.73%
  -> F1-Score                 :    4.57%
  -> ROC_AUC                  :   95.78%
------------------------------------------------------------
Resumo financeiro (KPI): $ 156,650

Análise de impacto financeiro
  Matriz de Confusão:
     [ TN:    13 | FP: 15612 ] -> Manutenções Corretas vs Desnecessárias
     [ FN:     1 | TP:   374 ] -> Falhas Perdidas vs Falhas Prevenidas
------------------------------------------------------------
  Detalhamento de Custos:
     -> Custos com Falsos Positivos (FP): R$  156,120
     -> Custos com Falsos Negativos (FN): R$      500

  Custo total da operação:         R$  156,620

Relatório de performance técnica
  -> Acurácia                 :    2.42%
  -> Precisão                 :    2.34%
  